In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
from scipy.stats import binom_test
from scipy.stats import binom
import argparse
import json
import configparser

np.seterr(invalid='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [23]:
DATABASE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_gpt_labelled.db"

TAG = "S"
CLASS = "n50"
OBL_TABLE = f"gpt_labelled_{TAG}_{CLASS}"


In [3]:

def examples_table(database, obl_table):
    """Gets all verb+comp+case plus tags and example sentences from spatial_obl (for hoverplot).
    Calculates
    - how many rows belong to each tag (ELT, A, S, or empty),
    - percentages per tag,
    - and up to 3 random example sentences for each tag+verbcase combination.
    Tables (not permanent):
    base: spatial obl join transaction_head. 0 or 1 for each tag if it is in tags column.
    aggregated: statistics. Groups by verbcase, what is the total count, count for each tag and count for no tag.
    long_counts: transforms wide counts into long format. Each row is verbcase,total,tag, count (each row has count for one tag).
    percentages: percentages for each tag in verbcase (based on long_counts).
    tagged_rows:  creates a normalized tag-level example table (from base).
    ranked_examples: randomly ranks/numbers example rows within each verbcase+tag.
    top_examples: takes 3 random examples per subgroup (verbcase+tag).
    examples_json: packages example rows into JSON arrays.
    final result: joins percentages and examples_json. 
    output format:
    verb	compound	case	tag	count	total	    pct	examples
    v1	    comp1	    ALL     A	    50	    100	    50.0	[...]
    v1	    comp1	    ALL     ELT	20	    100	    20.0	[...]
    
    """

    conn = sqlite3.connect(database)

    query = f"""
    WITH base AS (
        SELECT
            so.verb,
            so.verb_compound,
            so.morph_case,
            so.form,
            so.head_loc,
            so.sentence,
            so.sentence_id,
            th.phrase,
            th.form as verb_form,
            so.gpt_tags,

            CASE WHEN so.gpt_tags LIKE '%|ELT|%' THEN 1 ELSE 0 END AS is_elt,
            CASE WHEN so.gpt_tags LIKE '%|A|%'   THEN 1 ELSE 0 END AS is_a,
            CASE WHEN so.gpt_tags LIKE '%|S|%'   THEN 1 ELSE 0 END AS is_s,
            CASE WHEN so.gpt_tags = '' OR so.gpt_tags IS NULL THEN 1  ELSE 0 END AS is_empty

        FROM {obl_table} so

        LEFT JOIN transaction_head th
          ON so.sentence_id = th.sentence_id
         AND so.verb = th.verb
         AND so.verb_compound = th.verb_compound
        AND so.head_id = th.id
    ),

    aggregated AS (
        SELECT
            verb,
            verb_compound,
            morph_case,

            COUNT(*) AS total,

            SUM(is_elt)   AS elt_count,
            SUM(is_a)     AS a_count,
            SUM(is_s)     AS s_count,
            SUM(is_empty) AS empty_count

        FROM base

        GROUP BY
            verb,
            verb_compound,
            morph_case
    ),

    long_counts AS (

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'ELT' AS tag,
            elt_count AS count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'A',
            a_count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'S',
            s_count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            '' AS tag,
            empty_count
        FROM aggregated
    ),

    percentages AS (
        SELECT
            *,
            ROUND(100.0 * count / total, 2) AS pct
        FROM long_counts
    ),

    tagged_rows AS (

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'ELT' AS tag

        FROM base
        WHERE is_elt = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'A'

        FROM base
        WHERE is_a = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'S'

        FROM base
        WHERE is_s = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            '' AS tag

        FROM base
        WHERE is_empty = 1
    ),

    ranked_examples AS (
        SELECT
            verb,
            verb_compound,
            morph_case,
            tag,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,

            ROW_NUMBER() OVER (
                PARTITION BY
                    verb,
                    verb_compound,
                    morph_case,
                    tag
                ORDER BY RANDOM()
            ) AS rn

        FROM tagged_rows
    ),

    top_examples AS (
        SELECT *
        FROM ranked_examples
        WHERE rn <= 3
    ),

    examples_json AS (
        SELECT
            verb,
            verb_compound,
            morph_case,
            tag,
            -- information that is needed for highlighting/marking for plotting
            json_group_array(
                json_object(
                    'sentence', sentence,
                    'sentence_id', sentence_id,
                    'form', form,
                    'phrase', phrase,
                    'verb_form', verb_form,
                    'verb_compound', verb_compound,
                    'head_loc', head_loc
                )
            ) AS examples

        FROM top_examples

        GROUP BY
            verb,
            verb_compound,
            morph_case,
            tag
    )

    SELECT
        p.verb,
        p.verb_compound,
        p.morph_case,
        p.tag,
        p.count,
        p.total,
        p.pct,
        e.examples

    FROM percentages p

    LEFT JOIN examples_json e
      ON p.verb = e.verb
     AND p.verb_compound = e.verb_compound
     AND p.morph_case = e.morph_case
     AND p.tag = e.tag

    ORDER BY
        p.verb,
        p.verb_compound,
        p.morph_case,
        p.pct DESC;
    """

    df2 = pd.read_sql(query, conn)

    conn.close()

    return df2


In [24]:
examples_df = examples_table(DATABASE, OBL_TABLE)

In [25]:
examples_df

,verb,verb_compound,morph_case,tag,count,total,pct,examples
0,johtuma,,el,,160,319,50.16,"[{""sentence"":""Abilinnapea Ants Leemetsa selgit..."
1,johtuma,,el,ELT,78,319,24.45,"[{""sentence"":""Ma võin kinnitada see ei lõppe m..."
2,johtuma,,el,S,72,319,22.57,"[{""sentence"":""Mina oletan , et see johtus nend..."
3,johtuma,,el,A,9,319,2.82,"[{""sentence"":""Fatah' ametnikud ütlesid , et Qu..."
4,juhinduma,,el,,228,333,68.47,"[{""sentence"":""Ja toob näite , kuidas sellest t..."
5,juhinduma,,el,ELT,50,333,15.02,"[{""sentence"":""Valitsuse pressiteenistuse teate..."
6,juhinduma,,el,S,48,333,14.41,"[{""sentence"":""Ma oletan , et iga Riigikogu lii..."
7,juhinduma,,el,A,7,333,2.10,"[{""sentence"":""Rumsfeld juhindus jumalast"",""sen..."
8,kahtlema,,in,S,212,500,42.40,"[{""sentence"":""Kui toll ei kahtle hinna aktsept..."
9,kahtlema,,in,ELT,132,500,26.40,"[{""sentence"":""; Teisalt jälle kahtlesid paljud..."


In [26]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

examples_df.to_sql(f"gpt_labelled_{TAG}_{CLASS}_plotting_examples", conn, if_exists="replace", index=False)

conn.close()